# Dask Array (1)

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Dask"
* https://docs.dask.org/en/latest/array.html
* https://docs.dask.org/en/stable/array-chunks.html
* https://en.wikipedia.org/wiki/Taxicab_geometry
* https://docs.h5py.org/en/stable/

## Задачи для совместного разбора

1. Создайте массив размерностью 1000 на 300000, заполненный числами из стандартного нормального распределения. Исследуйте основные характеристики полученного массива.

2. Посчитайте сумму квадратов элементов массива, созданного в задаче 1. Создайте массив `np.array` такого же размера и сравните скорость решения задачи с использование `da.array` и `np.array`

## Лабораторная работа 7

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy`, `pandas` и `dask`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy`, структур `pandas` или структур `dask` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

В ходе выполнения все операции вычислений (расчет средних значений, расчет косинусной близости и т.д.) проводятся над `dask.array` и средствами пакета `dask`, если в задании не сказано обратного. Переход от `dask.array` к `numpy.array` или `pd.DataFrame` возможен исключительно для демонстрации результата в конце решения задачи. Если в задаче используются результаты выполнения предыдущих задач, то подразумевается, что вы используете результаты в виде `dask.array` (то есть то, что было получено до вызова `compute`, а не после).

<p class="task" id="1"></p>

1\. Считайте датасет `embeddings` из файла `recipe_embeddings.h5` в виде `dask.array`. Выведите на экран основную информацию о массиве: размер, количество векторов $M$ (количество строк в массиве), размерность каждого вектора $N$ (количество столбцов в массиве), тип, количество и размер сегментов.

In [ ]:
import h5py
import dask.array as da

In [ ]:
fp = h5py.File("./data/recipe_embeddings.h5")
arr = da.from_array(fp['embeddings'])
arr

<ipython-input-8-950ea435d2fe>:1: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  fp = h5py.File("./data/recipe_embeddings.h5")


dask.array<array, shape=(1200000, 312), dtype=float32, chunksize=(100000, 312), chunktype=numpy.ndarray>

<p class="task" id="2"></p>

2\. Посчитайте и выведите на экран среднее значение всех элементов массива. Исследуйте, как влияет значение аргумента `chunks` при создании `dask.array` на скорость выполнения операции поиска среднего.

Пусть $M$ - количество строк в массиве, $N$ - количество столбцов в массиве, `chunks=(r,c)`. Сравните несколько вариантов:
* $r=M$, $с \ll N$ ,
* $r \ll M$, $c=N$
* $r = M$, $c = N$
* значения $r, c$ по умолчанию.

Выберите наиболее оптимальные значения $r$ и  $c$ в смысле скорости вычислений и далее продолжайте работу с ними.

In [ ]:
%%time
arr = da.from_array(fp['embeddings'], chunks=(1200000, 30))
s = arr.mean()
s.compute()

Wall time: 4.17 s


0.002377757

In [ ]:
%%time
arr = da.from_array(fp['embeddings'], chunks=(1000, 312))
s = arr.mean()
s.compute()

Wall time: 1.46 s


0.0023777566

In [ ]:
%%time
arr = da.from_array(fp['embeddings'], chunks=(1200000, 312))
s = arr.mean()
s.compute()

Wall time: 852 ms


0.0023777678

In [ ]:
%%time
arr = da.from_array(fp['embeddings'])
s = arr.mean()
s.compute()

Wall time: 705 ms


0.0023777566

<p class="task" id="3"></p>

3\. Опишите пространство, в котором расположены эмбеддинги, посчитав минимальное и максимальное значение для каждой из координат. Сведите результаты в таблицу `pd.DataFrame`, состоящую из двух строк и 312 столбцов. Задайте индексы строк "min" и "max". Названия столбцов сделайте вида $e_i$. Выведите полученную таблицу на экран.

Решите задачу двумя способами. В первом варианте сделайте два вызова метода `compute` для расчета каждого из векторов максимальных и минимальных значений. Во втором варианте сделайте один вызов функции `dask.compute` для одновременного расчета двух векторов. Сравните время выполнения двух решений.

In [ ]:
import numpy as np
import pandas as pd
import dask

In [ ]:
%%time
min_ = arr.min(axis=0)
max_ = arr.max(axis=0)
pd.DataFrame([min_.compute(), max_.compute()], columns=['e' + str(i) for i in range(312)], index=['min', 'max'])

Wall time: 1.58 s


,e0,e1,e2,e3,e4,e5,e6,e7,e8,e9,...,e302,e303,e304,e305,e306,e307,e308,e309,e310,e311
min,-0.132803,-0.149056,-0.094468,-0.191697,-0.114229,-0.114341,-0.096039,-0.115178,-0.157275,-0.116715,...,-0.103254,-0.122285,-0.149789,-0.127703,-0.094802,-0.119690,-0.141425,-0.123732,-0.081543,-0.227348
max,0.135038,0.076125,0.157854,0.030987,0.101192,0.111774,0.147497,0.173821,0.099808,0.115573,...,0.119518,0.197589,0.113135,0.136490,0.162921,0.099021,0.086653,0.158176,0.166968,0.048967


In [ ]:
%%time
min_ = arr.min(axis=0)
max_ = arr.max(axis=0)
min_, max_ = dask.compute(min_, max_)
pd.DataFrame([min_, max_], columns=['e' + str(i) for i in range(312)], index=['min', 'max'])

Wall time: 876 ms


,e0,e1,e2,e3,e4,e5,e6,e7,e8,e9,...,e302,e303,e304,e305,e306,e307,e308,e309,e310,e311
min,-0.132803,-0.149056,-0.094468,-0.191697,-0.114229,-0.114341,-0.096039,-0.115178,-0.157275,-0.116715,...,-0.103254,-0.122285,-0.149789,-0.127703,-0.094802,-0.119690,-0.141425,-0.123732,-0.081543,-0.227348
max,0.135038,0.076125,0.157854,0.030987,0.101192,0.111774,0.147497,0.173821,0.099808,0.115573,...,0.119518,0.197589,0.113135,0.136490,0.162921,0.099021,0.086653,0.158176,0.166968,0.048967


<p class="task" id="4"></p>

4\. Датасет `embeddings` представляет собой набор 312-мерных векторов $x_i, i=0, 1, ... M-1$ Найдите вектор $x \ne x_{256}$ из набора данных, ближайший к вектору $x_{256}$ в смысле метрики $L_1$. Выведите на экран первые 10 координат вектора $x$.

$$d_1(\textbf{x},\textbf{y})=\sum_{k=1}^{n}{|x_i - y_i|}, \textbf{x}, \textbf{y} \in \mathbb{R}^n$$

In [ ]:
arr_ = (abs(arr.copy() - arr[256])).sum(axis=1)
arr_ = arr_[arr_.nonzero()]
arr_.compute_chunk_sizes()
x = arr[arr_.argmin() + 1 if arr_.argmin() >= 256 else arr_.argmin()]
x.compute()[:10]

C:\ProgramData\Anaconda3\lib\site-packages\dask\array\slicing.py:1076: PerformanceWarning: Increasing number of chunks by factor of 12
  p = blockwise(


array([ 0.0331987 , -0.03648246,  0.06629294, -0.0850755 , -0.04708353,
        0.00130241,  0.00259956,  0.01916818, -0.00985817, -0.04410348],
      dtype=float32)

<p class="task" id="5"></p>

5\. Рецепты разбиты на 4 группы. Загрузите маску для разбиения на группы из датасета `mask` из файла `recipe_embeddings.h5` в виде `dask.array`. Для каждой группы посчитайте и выведите на экран максимальное значение нормы $\ell_1$ векторов рецептов, принадлежащих к этой группе.

Подсказка: закодируйте маску принадлежности к группе при помощи метода кодирования one-hot encoding и воспользуйтесь механизмом распространения.

$$\ell_1: ||\textbf{x}||_1=\sum_{k=1}^{n}{|x_k|}, \textbf{x} \in \mathbb{R}^n$$

In [ ]:
mask = da.from_array(fp['mask']).reshape(1, 1200000)
mask_onehot = da.concatenate([mask == 0, mask == 1, mask == 2, mask == 3], axis=0)
mask_onehot

dask.array<concatenate, shape=(4, 1200000), dtype=bool, chunksize=(1, 1200000), chunktype=numpy.ndarray>

In [ ]:
max_l1_0 = (abs(arr[mask_onehot[0]])).sum(axis=1).max()
max_l1_1 = (abs(arr[mask_onehot[1]])).sum(axis=1).max()
max_l1_2 = (abs(arr[mask_onehot[2]])).sum(axis=1).max()
max_l1_3 = (abs(arr[mask_onehot[3]])).sum(axis=1).max()
dask.compute(max_l1_0, max_l1_1, max_l1_2, max_l1_3)

(13.319677, 13.324095, 13.31526, 13.319157)

<p class="task" id="6"></p>

6\. Работая с исходным файлом в формате `hdf`, реализуйте алгоритм подсчета среднего вектора датасета в блочной форме.

Блочный алгоритм вычислений состоит из двух частей:
1. Загрузка фрагмента за фрагментом данных и проведение вычислений над этим фрагментом
2. Агрегация результатов вычислений на различных фрагментах для получения результата на уровне всего набора данных

Важно: при работе с `hdf` в память загружаются не все элементы, а только те, которые запрашиваются в данный момент. При работе с `hdf` вы можете работать с массивами `numpy.array`. Для итерации по сегментам файла допускается использование циклов.

In [ ]:
embeddings = fp['embeddings']

In [ ]:
%%time
means = np.zeros((4, 312), dtype='float64')
for i, chunk in enumerate(np.array_split(embeddings, 4)):
    means[i] = chunk.mean(axis=0, dtype='float64') # вычисления для фрагмента
np.mean(means, axis=0, dtype='float64') # агрегация

Wall time: 1.06 s


array([ 9.75611741e-04, -3.39259919e-02,  4.66551852e-02, -8.74479048e-02,
       -9.71177284e-03,  5.51533102e-03,  2.71454279e-02,  3.75371801e-02,
       -2.35138376e-02, -1.10451345e-02,  2.61558130e-02,  8.86423980e-03,
        1.82164863e-02,  5.94255701e-02,  2.72058591e-02, -6.64570912e-03,
        3.04659187e-02,  1.03974861e-02,  1.93780558e-02,  1.43449808e-01,
       -2.46166348e-03, -5.94153239e-03, -3.18063951e-02, -4.10590974e-02,
        8.61882476e-02,  2.80386887e-02, -2.43214341e-02, -4.59721306e-03,
        1.09600983e-02,  3.78382257e-02, -1.06729240e-02, -1.57453825e-02,
        1.08159234e-03, -2.18519443e-02,  7.02830462e-03,  5.17747034e-02,
       -9.01944597e-04, -3.80986348e-02, -7.50486232e-02,  1.78839388e-02,
       -6.01361417e-02,  1.52847686e-01,  7.50525002e-02, -3.78862367e-02,
       -2.17200478e-02,  3.46801956e-03,  3.78258063e-02, -4.40569770e-02,
        4.40716660e-02, -3.56614161e-02,  8.02603242e-03, -3.51645609e-02,
       -4.23886347e-02,  

<p class="task" id="7"></p>

7\. Решите задачу 6, распараллелив вычисления при помощи `ThreadPool`. Сравните время и результаты решения работы вашего алгоритма с реализацией поиска среднего вектора из `dask`.

In [ ]:
from multiprocessing.pool import ThreadPool

In [ ]:
%%time
embeddings_split = np.array_split(embeddings, 4)
with ThreadPool(processes = 4) as pool:
    r = pool.map(lambda x: np.mean(x, axis=0, dtype='float64'), embeddings_split)
np.mean(r, axis=0, dtype='float64')

Wall time: 754 ms


array([ 9.75611741e-04, -3.39259919e-02,  4.66551852e-02, -8.74479048e-02,
       -9.71177284e-03,  5.51533102e-03,  2.71454279e-02,  3.75371801e-02,
       -2.35138376e-02, -1.10451345e-02,  2.61558130e-02,  8.86423980e-03,
        1.82164863e-02,  5.94255701e-02,  2.72058591e-02, -6.64570912e-03,
        3.04659187e-02,  1.03974861e-02,  1.93780558e-02,  1.43449808e-01,
       -2.46166348e-03, -5.94153239e-03, -3.18063951e-02, -4.10590974e-02,
        8.61882476e-02,  2.80386887e-02, -2.43214341e-02, -4.59721306e-03,
        1.09600983e-02,  3.78382257e-02, -1.06729240e-02, -1.57453825e-02,
        1.08159234e-03, -2.18519443e-02,  7.02830462e-03,  5.17747034e-02,
       -9.01944597e-04, -3.80986348e-02, -7.50486232e-02,  1.78839388e-02,
       -6.01361417e-02,  1.52847686e-01,  7.50525002e-02, -3.78862367e-02,
       -2.17200478e-02,  3.46801956e-03,  3.78258063e-02, -4.40569770e-02,
        4.40716660e-02, -3.56614161e-02,  8.02603242e-03, -3.51645609e-02,
       -4.23886347e-02,  

In [ ]:
%%time
embeddings_da = da.from_array(embeddings)
embeddings_da.mean(axis=0, dtype='float64').compute()

Wall time: 744 ms


array([ 9.75611741e-04, -3.39259919e-02,  4.66551852e-02, -8.74479048e-02,
       -9.71177284e-03,  5.51533102e-03,  2.71454279e-02,  3.75371801e-02,
       -2.35138376e-02, -1.10451345e-02,  2.61558130e-02,  8.86423980e-03,
        1.82164863e-02,  5.94255701e-02,  2.72058591e-02, -6.64570912e-03,
        3.04659187e-02,  1.03974861e-02,  1.93780558e-02,  1.43449808e-01,
       -2.46166348e-03, -5.94153239e-03, -3.18063951e-02, -4.10590974e-02,
        8.61882476e-02,  2.80386887e-02, -2.43214341e-02, -4.59721306e-03,
        1.09600983e-02,  3.78382257e-02, -1.06729240e-02, -1.57453825e-02,
        1.08159234e-03, -2.18519443e-02,  7.02830462e-03,  5.17747034e-02,
       -9.01944597e-04, -3.80986348e-02, -7.50486232e-02,  1.78839388e-02,
       -6.01361417e-02,  1.52847686e-01,  7.50525002e-02, -3.78862367e-02,
       -2.17200478e-02,  3.46801956e-03,  3.78258063e-02, -4.40569770e-02,
        4.40716660e-02, -3.56614161e-02,  8.02603242e-03, -3.51645609e-02,
       -4.23886347e-02,  